In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import PauliEvolutionGate

# Construction of the Hamiltonian
nqubits = 15 # You can change it
J_coupling = 1.0
h_field = 1.0

def ising_hamiltonian(n, J, h):
    paulis = []
    coeffs = []
    for i in range(n):
        for j in range(i+1, n):
            label = ["I"] * n
            label[i] = "Z"
            label[j] = "Z"
            paulis.append("".join(label))
            coeffs.append(J)
    for i in range(n):
        label = ["I"] * n
        label[i] = "X"
        paulis.append("".join(label))
        coeffs.append(h)
    return SparsePauliOp.from_list(list(zip(paulis, coeffs)))

Hamiltonian = ising_hamiltonian(nqubits, J_coupling, h_field)

H_matrix = Hamiltonian.to_matrix()
eigenvalues = eigh(H_matrix, eigvals_only=True)
exact_ground_energy = eigenvalues[0]

# Accurate Quantum Circuit and Quantum State Simulation
def build_ansatz(params, ops_list):
    qc = QuantumCircuit(nqubits)
    for op_str, param in zip(ops_list, params):
        op = SparsePauliOp(op_str)
        if len(op_str) == 1:
            for q in range(nqubits):
                qc.append(PauliEvolutionGate(op, time=param), [q])
        else:
            for q in range(nqubits - 1):
                qc.append(PauliEvolutionGate(op, time=param), [q, q+1])
    return qc

def get_energy(params, ops_list, H):
    qc = build_ansatz(params, ops_list)
    sv = Statevector(qc)
    return np.real(sv.expectation_value(H))

def get_V(params, ops_list, H, epsilon=1e-5):
    V = np.zeros(len(params))
    for i in range(len(params)):
        p_plus = np.copy(params)
        p_plus[i] += epsilon
        p_minus = np.copy(params)
        p_minus[i] -= epsilon
        V[i] = -(get_energy(p_plus, ops_list, H) - get_energy(p_minus, ops_list, H)) / (2 * epsilon)
    return V

def get_M(params, ops_list, epsilon=1e-5):
    sv_center = Statevector(build_ansatz(params, ops_list)).data
    derivatives = []
    for i in range(len(params)):
        p_plus = np.copy(params)
        p_plus[i] += epsilon
        p_minus = np.copy(params)
        p_minus[i] -= epsilon
        d_sv = (Statevector(build_ansatz(p_plus, ops_list)).data - Statevector(build_ansatz(p_minus, ops_list)).data) / (2 * epsilon)
        derivatives.append(d_sv)
        
    num_p = len(params)
    M = np.zeros((num_p, num_p))
    for i in range(num_p):
        for j in range(num_p):
            term1 = np.vdot(derivatives[i], derivatives[j])
            term2 = np.vdot(derivatives[i], sv_center) * np.vdot(sv_center, derivatives[j])
            M[i, j] = 2 * np.real(term1 + term2)
    return M

def get_residual(params, ops_list, H, M, V):
    e_val = get_energy(params, ops_list, H)
    H_sq = H.compose(H).simplify()
    e_sq_val = get_energy(params, ops_list, H_sq)
    var_H = e_sq_val - e_val**2
    
    M_reg = M + 0.01 * np.eye(len(params))
    M_inv = np.linalg.pinv(M_reg)
    residual = 2 * var_H - V.T @ M_inv @ V
    return residual

# Dynamic Growth Training Loop
def dynamic_vite_training(H, exact_energy, max_iters=60, learning_rate=0.05, threshold=0.05):
    CANDIDATE_OPS = ["XX", "YY", "ZZ", "XY", "YX", "XZ"]
    Ops = ["X", "Y", "Z"]
    current_params = np.random.uniform(-0.01, 0.01, len(Ops))
    
    energy_history = []
    depth_history = []
    residual_history = []
    
    print("Dynamic Variational Imaginary-Time Evolution Training")
    
    for iteration in range(max_iters):
        M = get_M(current_params, Ops)
        V = get_V(current_params, Ops, H)
        
        M_reg = M + 0.01 * np.eye(len(Ops))
        M_inv = np.linalg.pinv(M_reg)
        d_theta = M_inv @ V
        
        current_energy = get_energy(current_params, Ops, H)
        residual = get_residual(current_params, Ops, H, M, V)
        
        energy_history.append(current_energy)
        depth_history.append(len(Ops))
        residual_history.append(residual)
        
        print(f"Iteration {iteration+1:02d} Number of Parameters {len(Ops):02d} Energy {current_energy:.4f} Residual {residual:.4f}")
        
        if residual > threshold and len(CANDIDATE_OPS) > 0 and (current_energy - exact_energy) > 0.05:
            print("Residual Exceeds the Threshold and Energy Has Not Converged")
            new_op = CANDIDATE_OPS.pop(0)
            Ops.append(new_op)
            current_params = np.append(current_params, 0.0)
            print(f"Entangling Gates Have Been Dynamically Added {new_op}")
        else:
            current_params = current_params + learning_rate * d_theta
            
    return current_params, energy_history, depth_history, residual_history

# Execution and Plotting
final_params, energies, depths, residuals = dynamic_vite_training(Hamiltonian, exact_ground_energy)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 10))

ax1.plot(energies, marker='o', label='Energy', color='blue')
ax1.axhline(y=exact_ground_energy, color='gray', linestyle='--', label=f'Exact Ground State ({Exact_ground_energy:.4f})')
ax1.set_ylabel('Energy (Hartree)')
ax1.set_title('Dynamic Ansatz VITE Optimization')
ax1.legend()

ax2.plot(depths, marker='s', label='Parameter Count', color='green', drawstyle='steps-post')
ax2.set_ylabel('Parameter Count')
ax2.legend()

ax3.plot(residuals, marker='^', label='McLachlan Residual', color='red')
ax3.set_ylabel('Residual')
ax3.set_xlabel('Iterations')
ax3.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Resource Comparison: N Qubits, 50 Iterations
nqubits = 3  # You can change it
iters = 50

methods = [
    "Static GD",
    "Static VITE",
    "ADAPT-VQE",
    "Dynamic VITE"
]

# Number of parameters (based on the logic of your code)
initial_params = [9, 9, 1, 3]
max_params = [9, 9, 51, 12]

# Number of Two-Qubit Operators
# Static methods use 6 fixed global two-qubit operators (XX, YY, ZZ, XY, YX, XZ)
# ADAPT-VQE starts from 0 and adds two-qubit operators at all 50 steps
# Dynamic VITE starts from 0 and adds up to 9 two-qubit candidate operators
initial_two_qubit_ops = [6, 6, 0, 0]
max_two_qubit_ops = [6, 6, 50, 9]


# Adaptive Topology CNOT Calculation (Fully Connected)

# 1. Calculate the number of pairs in a fully connected topology: N(N-1)/2
# 2. Each pair requires 2 CNOT gates to implement Pauli evolution
# 3. Total CNOT gates required for each global two-qubit operator:
#    2 * (N(N-1)/2) = N(N-1)
cnot_per_global_op = nqubits * (nqubits - 1)

initial_cnot = [cnot_per_global_op * x for x in initial_two_qubit_ops]
max_cnot = [cnot_per_global_op * x for x in max_two_qubit_ops]


# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

colors = ["#E69F00", "#009E73", "#CC79A7", "#0072B2"]
x = np.arange(len(methods))
width = 0.58

# ---- Plot 1: Number of parameters ----
ax = axes[0]

bars = ax.bar(
    x, initial_params, width,
    color=colors, alpha=0.85,
    label="Initial parameters"
)

# Show growth from initial to maximum
for i in range(len(methods)):
    if max_params[i] > initial_params[i]:
        ax.bar(
            x[i],
            max_params[i] - initial_params[i],
            width,
            bottom=initial_params[i],
            color=colors[i],
            alpha=0.30,
            hatch="//",
            edgecolor=colors[i],
            linewidth=1.2
        )

    ax.text(
        x[i], max_params[i] + 1.0,
        f"{initial_params[i]} -> {max_params[i]}",
        ha="center", va="bottom", fontsize=10
    )

ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=15, ha="right")
ax.set_ylabel("Number of variational parameters")
ax.set_title("Parameter Count")
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.set_ylim(0, max(max_params) * 1.18)

# ---- Plot 2: CNOT count per circuit ----
ax = axes[1]

for i in range(len(methods)):
    lower = initial_cnot[i]
    upper = max_cnot[i]

    ax.bar(
        x[i], lower, width,
        color=colors[i], alpha=0.85
    )

    if upper > lower:
        ax.bar(
            x[i], upper - lower, width,
            bottom=lower,
            color=colors[i], alpha=0.30,
            hatch="//",
            edgecolor=colors[i],
            linewidth=1.2
        )

    if lower == upper:
        label = f"{lower}"
    else:
        label = f"{lower}–{upper}"

    ax.text(
        x[i], upper + (max(max_cnot) * 0.05), # Dynamically adjust label height
        label,
        ha="center", va="bottom", fontsize=10
    )

ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=15, ha="right")
ax.set_ylabel("Estimated CNOT gates per circuit")
ax.set_title(f"CNOT Resource Estimate (Fully Connected, N={nqubits})")
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.set_ylim(0, max(max_cnot) * 1.22)

fig.suptitle(
    f"Quantum Resource Comparison (System Size: {nqubits} Qubits)",
    fontsize=14
)

fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy.linalg import eigh
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import PauliEvolutionGate
from functools import lru_cache


# System Settings

N_QUBITS = 10 # You can change it
J_COUPLING = 1.0
H_FIELD = 1.0

ITERS = 50
NUM_RUNS = 5
LEARNING_RATE = 0.05
EPSILON = 1e-5
REGULARIZATION = 0.01
BASE_SEED = 2026

FIXED_OPS = [
    "X", "Y", "Z",
    "XX", "YY", "ZZ",
    "XY", "YX", "XZ"
]

COLORS = {
    "Static GD": "orange",
    "Static VITE": "green",
    "ADAPT-VQE": "purple",
    "Dynamic VITE": "blue",
}


def ising_hamiltonian(n, J, h):
    paulis, coeffs = [], []

    for i in range(n):
        for j in range(i + 1, n):
            label = ["I"] * n
            label[i] = "Z"
            label[j] = "Z"
            paulis.append("".join(label))
            coeffs.append(J)

    for i in range(n):
        label = ["I"] * n
        label[i] = "X"
        paulis.append("".join(label))
        coeffs.append(h)

    return SparsePauliOp.from_list(
        list(zip(paulis, coeffs))
    )


HAMILTONIAN = ising_hamiltonian(
    N_QUBITS, J_COUPLING, H_FIELD
)

H_MATRIX = HAMILTONIAN.to_matrix()
EXACT_ENERGY = float(
    eigh(H_MATRIX, eigvals_only=True)[0]
)

print(f"Exact ground energy: {EXACT_ENERGY:.10f}")


# Ansatz

def build_ansatz(params, ops_list):
    qc = QuantumCircuit(N_QUBITS)

    for op_str, param in zip(ops_list, params):
        op = SparsePauliOp(op_str)

        if len(op_str) == 1:
            for q in range(N_QUBITS):
                qc.append(
                    PauliEvolutionGate(op, time=float(param)),
                    [q]
                )
        else:
            for q in range(N_QUBITS - 1):
                qc.append(
                    PauliEvolutionGate(op, time=float(param)),
                    [q, q + 1]
                )

    return qc


# Resource Tracking
RESOURCE = {
    "evaluations": 0,
    "cnot_total": 0,
}


def reset_resources():
    RESOURCE["evaluations"] = 0
    RESOURCE["cnot_total"] = 0


@lru_cache(maxsize=None)
def count_cnot(ops_tuple):
    """
    Count CNOT gates after transpilation for this ansatz structure.
    Use representative nonzero parameters to avoid zero-angle
    gates disappearing from the resource estimate.
    """
    params = np.full(len(ops_tuple), 0.173)

    qc = build_ansatz(params, list(ops_tuple))

    compiled = transpile(
        qc,
        basis_gates=["u", "cx"],
        optimization_level=0
    )

    return int(compiled.count_ops().get("cx", 0))


def tracked_statevector(params, ops_list):
    qc = build_ansatz(params, ops_list)

    RESOURCE["evaluations"] += 1
    RESOURCE["cnot_total"] += count_cnot(tuple(ops_list))

    return Statevector(qc)


# Energy and Gradient

def get_energy(params, ops_list, H):
    sv = tracked_statevector(params, ops_list)
    return float(np.real(sv.expectation_value(H)))


def get_V(params, ops_list, H, epsilon=EPSILON):
    V = np.zeros(len(params))

    for i in range(len(params)):
        p_plus = params.copy()
        p_minus = params.copy()

        p_plus[i] += epsilon
        p_minus[i] -= epsilon

        e_plus = get_energy(p_plus, ops_list, H)
        e_minus = get_energy(p_minus, ops_list, H)

        V[i] = -(e_plus - e_minus) / (2 * epsilon)

    return V


# Quantum Geometric Matrix

def get_M(params, ops_list, epsilon=EPSILON):
    sv_center = tracked_statevector(
        params, ops_list
    ).data

    derivatives = []

    for i in range(len(params)):
        p_plus = params.copy()
        p_minus = params.copy()

        p_plus[i] += epsilon
        p_minus[i] -= epsilon

        sv_plus = tracked_statevector(
            p_plus, ops_list
        ).data

        sv_minus = tracked_statevector(
            p_minus, ops_list
        ).data

        derivatives.append(
            (sv_plus - sv_minus) / (2 * epsilon)
        )

    p = len(params)
    M = np.zeros((p, p))

    for i in range(p):
        for j in range(p):
            term1 = np.vdot(derivatives[i], derivatives[j])
            term2 = (
                np.vdot(derivatives[i], sv_center)
                * np.vdot(sv_center, derivatives[j])
            )
            M[i, j] = 2 * np.real(term1 + term2)

    return M


# Residual

def get_residual(params, ops_list, H, M, V):
    energy = get_energy(params, ops_list, H)
    H_squared = (H @ H).simplify()
    energy_squared = get_energy(params, ops_list, H_squared)

    variance = max(0.0, energy_squared - energy**2)

    M_reg = M + REGULARIZATION * np.eye(len(params))
    M_inv = np.linalg.pinv(M_reg)

    return float(2 * variance - V.T @ M_inv @ V)


# Static GD
def static_gd_training(H, max_iters=ITERS,
                       learning_rate=LEARNING_RATE):
    params = np.random.uniform(-0.1, 0.1, len(FIXED_OPS))
    history = []

    for _ in range(max_iters):
        V = get_V(params, FIXED_OPS, H)
        params = params + learning_rate * V
        history.append(get_energy(params, FIXED_OPS, H))

    return history


# Static VITE
def static_vite_training(H, max_iters=ITERS,
                         learning_rate=LEARNING_RATE):
    params = np.random.uniform(-0.1, 0.1, len(FIXED_OPS))
    history = []

    for _ in range(max_iters):
        M = get_M(params, FIXED_OPS)
        V = get_V(params, FIXED_OPS, H)

        M_reg = M + REGULARIZATION * np.eye(len(FIXED_OPS))
        dtheta = np.linalg.pinv(M_reg) @ V

        params = params + learning_rate * dtheta
        history.append(get_energy(params, FIXED_OPS, H))

    return history


# ADAPT-VQE

def adapt_vqe_training(H, max_iters=ITERS,
                       learning_rate=LEARNING_RATE):
    pool = [
        "X", "Y", "Z",
        "XX", "YY", "ZZ",
        "XY", "YX", "XZ"
    ]

    ops = ["X"]
    params = np.random.uniform(-0.01, 0.01, len(ops))
    history = []

    for _ in range(max_iters):
        best_grad = -1.0
        best_op = pool[0]

        for op in pool:
            candidate_ops = ops + [op]
            candidate_params = np.append(params, 0.0)

            V = get_V(candidate_params, candidate_ops, H)
            grad_magnitude = abs(V[-1])

            if grad_magnitude > best_grad:
                best_grad = grad_magnitude
                best_op = op

        ops.append(best_op)
        params = np.append(params, 0.0)

        for _ in range(3):
            V = get_V(params, ops, H)
            params = params + learning_rate * V

        history.append(get_energy(params, ops, H))

    return history


# Dynamic VITE
def dynamic_vite_training(H, exact_energy,
                          max_iters=ITERS,
                          learning_rate=LEARNING_RATE,
                          threshold=0.05):
    candidates = [
        "XX", "YY", "ZZ",
        "XY", "YX", "XZ",
        "ZX", "YZ", "ZY"
    ]

    ops = ["X", "Y", "Z"]
    params = np.random.uniform(-0.01, 0.01, len(ops))
    history = []

    for _ in range(max_iters):
        M = get_M(params, ops)
        V = get_V(params, ops, H)

        M_reg = M + REGULARIZATION * np.eye(len(ops))
        dtheta = np.linalg.pinv(M_reg) @ V

        energy = get_energy(params, ops, H)
        residual = get_residual(params, ops, H, M, V)

        if (
            residual > threshold
            and candidates
            and energy - exact_energy > 0.05
        ):
            new_op = candidates.pop(0)
            ops.append(new_op)
            params = np.append(params, 0.0)
        else:
            params = params + learning_rate * dtheta

        history.append(get_energy(params, ops, H))

    return history


# Run All Algorithms
algorithms = {
    "Static GD": static_gd_training,
    "Static VITE": static_vite_training,
    "ADAPT-VQE": adapt_vqe_training,
    "Dynamic VITE": dynamic_vite_training,
}

results = []
histories = {name: [] for name in algorithms}

for method_index, (name, train_func) in enumerate(algorithms.items()):
    for run in range(NUM_RUNS):
        np.random.seed(BASE_SEED + 100 * method_index + run)
        reset_resources()

        if name == "Dynamic VITE":
            history = train_func(
                HAMILTONIAN, EXACT_ENERGY, max_iters=ITERS
            )
        else:
            history = train_func(
                HAMILTONIAN, max_iters=ITERS
            )

        final_energy = float(history[-1])
        energy_error = abs(final_energy - EXACT_ENERGY)

        results.append({
            "Method": name,
            "Run": run + 1,
            "Final Energy": final_energy,
            "Energy Error": energy_error,
            "Circuit Evaluations": RESOURCE["evaluations"],
            "Cumulative CNOT": RESOURCE["cnot_total"],
        })

        histories[name].append(history)

        print(
            f"{name:12s} | Run {run + 1}/{NUM_RUNS} | "
            f"Error={energy_error:.6e} | "
            f"Evaluations={RESOURCE['evaluations']} | "
            f"CNOT={RESOURCE['cnot_total']}"
        )


# Save CSV and Summary

df = pd.DataFrame(results)
df.to_csv("resource_benchmark_runs.csv", index=False)

summary = df.groupby("Method").agg({
    "Final Energy": ["mean", "std"],
    "Energy Error": ["mean", "std"],
    "Circuit Evaluations": ["mean", "std"],
    "Cumulative CNOT": ["mean", "std"],
})

summary.to_csv("resource_benchmark_summary.csv")

print("\n=== Summary ===")
print(summary.round(6))


# Three-Metric Comparison Plot
methods = list(algorithms.keys())

plt.figure(figsize=(10, 6))

for i, method in enumerate(methods):
    values = df.loc[
        df["Method"] == method,
        "Circuit Evaluations"
    ].to_numpy()

    mean_value = values.mean()
    std_value = (
        values.std(ddof=1)
        if len(values) > 1 else 0.0
    )

    plt.bar(
        i,
        mean_value,
        yerr=std_value,
        color=COLORS[method],
        alpha=0.8,
        capsize=5
    )

    plt.scatter(
        np.full(len(values), i),
        values,
        color="black",
        s=25,
        zorder=3
    )

plt.ylabel("Cumulative Circuit Evaluations")
plt.title("Cumulative Circuit Evaluations Comparison")

plt.xticks(
    range(len(methods)),
    methods,
    rotation=20,
    ha="right"
)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

plt.tight_layout()

plt.show()
plt.close()


# Cumulative CNOT Count Plot

plt.figure(figsize=(10, 6))

for i, method in enumerate(methods):
    values = df.loc[
        df["Method"] == method,
        "Cumulative CNOT"
    ].to_numpy()

    mean_value = values.mean()
    std_value = (
        values.std(ddof=1)
        if len(values) > 1 else 0.0
    )

    plt.bar(
        i,
        mean_value,
        yerr=std_value,
        color=COLORS[method],
        alpha=0.8,
        capsize=5
    )

    plt.scatter(
        np.full(len(values), i),
        values,
        color="black",
        s=25,
        zorder=3
    )

plt.ylabel("Cumulative CNOT Count")
plt.title("Cumulative CNOT Count Comparison")

plt.xticks(
    range(len(methods)),
    methods,
    rotation=20,
    ha="right"
)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

plt.tight_layout()


plt.show()
plt.close()


# Energy Convergence Plot

plt.figure(figsize=(10, 6))

for method in methods:
    data = np.asarray(histories[method])

    mean_energy = data.mean(axis=0)
    std_energy = data.std(axis=0)

    iteration_axis = np.arange(
        1, data.shape[1] + 1
    )

    plt.plot(
        iteration_axis,
        mean_energy,
        color=COLORS[method],
        label=method
    )

    plt.fill_between(
        iteration_axis,
        mean_energy - std_energy,
        mean_energy + std_energy,
        color=COLORS[method],
        alpha=0.15
    )

plt.axhline(
    EXACT_ENERGY,
    color="black",
    linestyle="--",
    label=f"Exact Ground Energy ({EXACT_ENERGY:.4f})"
)

plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.title("Energy Convergence Comparison")

plt.grid(
    True,
    linestyle="--",
    alpha=0.3
)

plt.legend()
plt.tight_layout()

plt.show()
plt.close()